# ENEC 2026 · Optional extension 2 · A-001: Anomaly Detection and 7-Day Vibration Forecast

**Time:** about 60 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Optional homework or early-finisher work. Use 90 days of hourly sensor data to spot when A-001 changed, then forecast the next 7 days with a simple, explainable model.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai 

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## Business question

> Is A-001's vibration likely to keep rising, so the reliability team can plan inspection earlier?

This notebook supports a human decision. It does **not** decide to stop, repair or replace anything.

**Flow:** hourly data → quality check → look → anomaly flags → features → chronological split → baseline → Ridge model → backtest → forecast → bounded interpretation.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error

df = pd.read_csv(enec.CSV_DIR / "a001_sensor_hourly_90d.csv", parse_dates=["hour_timestamp"])
df = df.sort_values("hour_timestamp").reset_index(drop=True)
print(df.shape, df.hour_timestamp.min(), "->", df.hour_timestamp.max())
df.head()

## 1. Data quality first

In [ ]:
VARIABLES = enec.VARIABLES
print("Duplicate timestamps:", df.hour_timestamp.duplicated().sum())
print("Missing values:\n", df[VARIABLES].isna().sum())
expected = pd.date_range(df.hour_timestamp.min(), df.hour_timestamp.max(), freq="h")
print("Missing hours:", len(expected.difference(df.hour_timestamp)))
print("Hours with sensor quality issues:", int((df.quality_issue_minutes > 0).sum()))

## 2. Look at the pattern, not just the latest value

In [ ]:
first_anom = df.loc[df.anomaly_minutes > 0, "hour_timestamp"].min()
fig, ax = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax[0].plot(df.hour_timestamp, df.avg_vibration_mm_s, color="#22d3ee"); ax[0].set_ylabel("Vibration (mm/s)")
ax[1].plot(df.hour_timestamp, df.avg_temperature_c, color="#f5a623"); ax[1].set_ylabel("Temperature (C)")
for a in ax:
    a.grid(alpha=.25)
    if pd.notna(first_anom):
        a.axvline(first_anom, color="red", ls="--", alpha=.6)
ax[0].set_title("A-001 hourly averages (red line = first hour with anomaly minutes)")
plt.tight_layout(); plt.show()
print("First hour with anomaly minutes:", first_anom)

In [ ]:
df[VARIABLES].corr().round(2)   # correlation is not causation: it only shows what moves together

## 3. Simple anomaly detection

Two beginner-friendly ideas:

1. **Baseline z-score:** how far is each hour from the first 45 days (a "normal" window)?
2. **Isolation Forest:** an unsupervised model that isolates unusual combinations of sensors.

In [ ]:
base = df.iloc[: 24 * 45]
mu, sd = base.avg_vibration_mm_s.mean(), base.avg_vibration_mm_s.std()
df["vib_z"] = (df.avg_vibration_mm_s - mu) / sd
df["z_flag"] = df.vib_z.abs() > 4

from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.05, random_state=42)
df["iso_flag"] = iso.fit_predict(df[VARIABLES]) == -1

print("z-score flags  :", int(df.z_flag.sum()), "hours")
print("IsolationForest:", int(df.iso_flag.sum()), "hours")
print("First z-flag   :", df.loc[df.z_flag, "hour_timestamp"].min())

plt.figure(figsize=(14, 4))
plt.plot(df.hour_timestamp, df.avg_vibration_mm_s, color="#8b5cf6", lw=1)
plt.scatter(df.loc[df.z_flag, "hour_timestamp"], df.loc[df.z_flag, "avg_vibration_mm_s"], s=8, color="red", label="z-score flag")
plt.title("Vibration with baseline z-score flags"); plt.legend(); plt.grid(alpha=.25); plt.show()

**Discuss:** A flag is a *question*, not an answer. What would you check before deciding the pump has a problem? (Sensor quality, recent maintenance, operating state, other sensors.)

## 4. Time-series features

A normal regression model does not understand time. We give it a little memory: the value 1 hour ago, 24 hours ago, the 24-hour rolling mean, and the hour of day / day of week as sine and cosine.

In [ ]:
import inspect
print(inspect.getsource(enec.make_features))

In [ ]:
model_df, FEATURES = enec.make_features(df)
print(len(model_df), "rows,", len(FEATURES), "features")

## 5. Chronological split (never shuffle a time series)

We keep the **last 7 days (168 hours)** completely unseen for the test.

In [ ]:
H = 24 * 7
train, test = model_df.iloc[:-H], model_df.iloc[-H:]
print("train rows:", len(train), "| test rows:", len(test))
print("test period:", test.hour_timestamp.min(), "->", test.hour_timestamp.max())

## 6. A baseline to beat: "next week looks like yesterday"

In [ ]:
y_true = test.avg_vibration_mm_s.to_numpy()
baseline = np.tile(train.avg_vibration_mm_s.iloc[-24:].to_numpy(), 7)
base_mae = mean_absolute_error(y_true, baseline)
base_rmse = np.sqrt(mean_squared_error(y_true, baseline))
print(f"Baseline  MAE {base_mae:.3f}  RMSE {base_rmse:.3f}")

## 7. Multivariate Ridge model, backtested on the unseen week

In [ ]:
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(train[FEATURES], train[VARIABLES])
history = df[df.hour_timestamp < test.hour_timestamp.min()]
back = enec.recursive_forecast(model, history, H)

mae = mean_absolute_error(y_true, back.avg_vibration_mm_s)
rmse = np.sqrt(mean_squared_error(y_true, back.avg_vibration_mm_s))
print(f"Ridge     MAE {mae:.3f}  RMSE {rmse:.3f}   (baseline MAE {base_mae:.3f})")

plt.figure(figsize=(14, 4.5))
plt.plot(test.hour_timestamp, y_true, label="Actual")
plt.plot(back.hour_timestamp, back.avg_vibration_mm_s, label="Ridge backtest")
plt.plot(test.hour_timestamp, baseline, label="Repeat-yesterday baseline", alpha=.6)
plt.title("A-001 vibration: 7-day backtest"); plt.legend(); plt.grid(alpha=.25); plt.show()

**Read the metrics honestly.** MAE is the average absolute error (lower is better). If Ridge only slightly beats the baseline, say so. The last week of this dataset is the volatile one, which is exactly where simple models struggle. A good analyst reports that, not just the best-looking number.

## 8. Retrain on all history and forecast the next 7 days

In [ ]:
final = make_pipeline(StandardScaler(), Ridge(alpha=1.0)).fit(model_df[FEATURES], model_df[VARIABLES])
future = enec.recursive_forecast(final, df, H)

recent = df.tail(24 * 14)
plt.figure(figsize=(14, 4.5))
plt.plot(recent.hour_timestamp, recent.avg_vibration_mm_s, label="Last 14 days (actual)")
plt.plot(future.hour_timestamp, future.avg_vibration_mm_s, label="Next 7 days (forecast)", color="#f5a623")
plt.title("A-001 vibration: next 7 days"); plt.legend(); plt.grid(alpha=.25); plt.show()

recent_mean = df.avg_vibration_mm_s.tail(H).mean()
future_mean = future.avg_vibration_mm_s.mean()
summary = {"recent_7d_mean_mm_s": round(recent_mean, 3), "forecast_next_7d_mean_mm_s": round(future_mean, 3),
           "forecast_change_pct": round(100 * (future_mean - recent_mean) / recent_mean, 1),
           "backtest_mae_mm_s": round(mae, 3), "baseline_mae_mm_s": round(base_mae, 3)}
summary

## 9. Bounded interpretation with an LLM

We give the model **numbers we computed**, not raw data, and a strict boundary. The model writes the words; the code did the maths.

In [ ]:
system = ("You write a short, plain-English note for a reliability reviewer. Use ONLY the numbers given. "
          "Mention the forecast error honestly. Do not invent thresholds. Do not diagnose a failure mode. "
          "Do not recommend stopping or repairing equipment. End with one suggested human check.")
print(enec.ask(client, "NUMBERS: " + str(summary), system=system))

## Limits to say out loud

- Synthetic data. A real plant needs real thresholds, operating context and engineering judgement.
- A forecast extrapolates a pattern. It does **not** prove a cause or a failure.
- A 90-day window is small. A full year of hourly data would show the seasonal pattern better.

## Exercise

1. Change `base` to the first 30 days. How many z-flags do you get now?
2. Add a 168-hour lag feature. Does the backtest MAE improve?
3. Forecast `avg_temperature_c` instead. Is the model more or less accurate than for vibration? Why might that be?